In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from imblearn.over_sampling import SMOTE


In [ ]:
def dataset_smote(beta, ph):
    # Import
    X_df = pd.read_csv(beta, delimiter=' ')
    y_df = pd.read_csv(ph, delimiter=' ')
    X = X_df.values
    y = y_df['label'].values

    # SMOTE
    sm = SMOTE(random_state=42)  # Set random seed for reproducibility
    X_res, y_res = sm.fit_resample(X, y)
    X_res = np.round(X_res, decimals=5)
    y_res = to_categorical(y_res)

    # Print the class distribution before and after SMOTE
    print("Original class distribution:")
    print({i: np.sum(y == i) for i in np.unique(y)})
    print("Class distribution after SMOTE:")
    print({i: np.sum(np.argmax(y_res, axis=1) == i) for i in range(y_res.shape[1])})
    return X_df, X_res, y_res

def nn_model(input_shape, num_classes):
    model = Sequential([
        Dense(64, input_dim=input_shape, activation='relu'),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def skf_process(X, y, n_splits=10):
    # Step 4: Set up StratifiedKFold for cross-validation
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # Initialize lists to store metrics for each fold
    accuracy_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []

    fold_no = 1

    # Step 5: Train and evaluate the model for each fold
    for train_index, test_index in skf.split(X, np.argmax(y, axis=1)):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        # Create a new model instance
        model = nn_model(input_shape=X.shape[1], num_classes=y.shape[1])

        # Train the model
        history = model.fit(X_train, y_train, epochs=50, batch_size=20, verbose=0)

        # Evaluate the model
        scores = model.evaluate(X_test, y_test, verbose=0)
        print(f'Score for fold {fold_no}: {model.metrics_names[1]} of {scores[1]*100}%')
        accuracy_scores.append(scores[1])

        # Predict the classes for the test set
        y_pred = np.argmax(model.predict(X_test), axis=1)
        y_test_labels = np.argmax(y_test, axis=1)

        # Calculate evaluation metrics
        accuracy = accuracy_score(y_test_labels, y_pred)
        precision = precision_score(y_test_labels, y_pred, average='weighted')
        recall = recall_score(y_test_labels, y_pred, average='weighted')
        f1 = f1_score(y_test_labels, y_pred, average='weighted')

        # Append the metrics to the respective lists
        accuracy_scores.append(accuracy)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)

        # Print metrics for the current fold
        print(f"Fold {len(accuracy_scores)}:")
        print(f"Accuracy: {accuracy:.3f}")
        print(f"Precision: {precision:.3f}")
        print(f"Recall: {recall:.3f}")
        print(f"F1 Score: {f1:.3f}")
        print("-" * 30)

        fold_no += 1

    return {
        "accuracy": accuracy_scores,
        "precision": precision_scores,
        "recall": recall_scores,
        "f1": f1_scores
    }

def avg_std(metrics):
    """
    Calculate and print the average and standard deviation of metrics.

    Parameters:
    metrics: A dictionary containing lists of metrics for each fold.
    """
    avg_accuracy = np.mean(metrics['accuracy'])
    std_accuracy = np.std(metrics['accuracy'])
    avg_precision = np.mean(metrics['precision'])
    std_precision = np.std(metrics['precision'])
    avg_recall = np.mean(metrics['recall'])
    std_recall = np.std(metrics['recall'])
    avg_f1 = np.mean(metrics['f1'])
    std_f1 = np.std(metrics['f1'])

    print("\nAverage Scores Across all Folds:")
    print(f"Accuracy: {avg_accuracy:.3f} +- {std_accuracy:.3f}")
    print(f"Precision: {avg_precision:.3f} +- {std_precision:.3f}")
    print(f"Recall: {avg_recall:.3f} +- {std_recall:.3f}")
    print(f"F1-score: {avg_f1:.3f} +- {std_f1:.3f}")

# Example usage:


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/TA_KIM/data')
dir = os.getcwd()
beta = f'{dir}/beta_p56_01_noppi.txt'
ph = f'{dir}/ph_label_universal.txt'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
X_df, X, y = dataset_smote(beta, ph)



Original class distribution:
{0: 100, 1: 46, 2: 167, 3: 120}
Class distribution after SMOTE:
{0: 167, 1: 167, 2: 167, 3: 167}


In [ ]:
metrics = skf_process(X, y)
# avg_std(metrics)

Score for fold 1: accuracy of 88.05969953536987%
3/3 [==============================] - 0s 3ms/step
Fold 2:
Accuracy: 0.881
Precision: 0.884
Recall: 0.881
F1 Score: 0.882
------------------------------
Score for fold 2: accuracy of 85.07462739944458%
3/3 [==============================] - 0s 4ms/step
Fold 4:
Accuracy: 0.851
Precision: 0.851
Recall: 0.851
F1 Score: 0.848
------------------------------
Score for fold 3: accuracy of 83.58209133148193%
3/3 [==============================] - 0s 5ms/step
Fold 6:
Accuracy: 0.836
Precision: 0.859
Recall: 0.836
F1 Score: 0.822
------------------------------
Score for fold 4: accuracy of 83.58209133148193%
3/3 [==============================] - 0s 5ms/step
Fold 8:
Accuracy: 0.836
Precision: 0.828
Recall: 0.836
F1 Score: 0.830
------------------------------
Score for fold 5: accuracy of 88.05969953536987%
3/3 [==============================] - 0s 4ms/step
Fold 10:
Accuracy: 0.881
Precision: 0.881
Recall: 0.881
F1 Score: 0.881
--------------------

In [ ]:
avg_std(metrics)


Average Scores Across all Folds:
Accuracy: 0.817 +- 0.031
Precision: 0.821 +- 0.035
Recall: 0.817 +- 0.031
F1-score: 0.809 +- 0.036
